# Convolutional Neural Network Block
- Add-ons: Basic, Optimization, Efficiency, Mobile, Attention
- Basic: Conv -> Norm -> Activation
- Change channel, preserve spatial resolution with padding
- Early and middle stages of CNNs
- Good default feature extraction, stable training. 
- FLOP cost grows with C_in x C_out 

In [ ]:
import torch
import torch.nn as nn

class BasicConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.layers(x)

# Residual Block
- Skip connection -> Match shape
- If channel/stride changes, use 1 x 1 conv for shortcut
- Useful: Deep network (default for deep CNNs)
- Still need norm/regularization
- Basic Layout: Conv -> Norm -> Conv -> Norm 
                  |--- 1 x 1 Conv --------|

In [ ]:
import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.BatchNorm2d(out_channels),
            nn.Conv2d(out_channels, out_channels, kernel_size, stride, padding),
            nn.BatchNorm2d(out_channels)
        )
        self.shortcut = nn.Identity() if in_channels == out_channels else nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
            nn.BatchNorm2d(out_channels)
        )
    
    def forward(self, x):
        out = self.layers(x)
        out += self.shortcut(x)
        return torch.relu(out)

# Bottleneck
- Standard Conv: C_in x C_out x H x W
- Bottleneck: C_in x C_mid + C_mid x C_mid x H x W + C_mid x C_out
- C_mid << C_in | C_mid << C_out
- Best: Deep CNNs, high channel stages when 3 x 3 is expensive
- Deep network but fixed FLOP budget
- If C_mid too small, representational capacity bottleneck accuracy
- Extra 1 x 1 add memory overhead instead of faster

In [ ]:
import torch
import torch.nn as nn

class BottleNeckBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super().__init__()
        mid_channels = out_channels // 4
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=1),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, mid_channels, kernel_size=kernel_size, stride=stride, padding=padding),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=1),
            nn.BatchNorm2d(out_channels)
        )
    
    def forward(self, x):
        return self.layers(x)

# Depthwise Separable
- Standard: C_out x C_in x H_K x W_K + C_out
- Depthwise seperable: (C_in x H_K x W_K + C_in) + (C_out x C_in + C_out) <<< Standard
- Focus on reduce parameters when k is large
- Use on edge devices, real-time system, large-scale deployment
- Reduce accuracy if channel mixing is weak.
- Pointwise conv cost still dominates at large channel sizes

In [ ]:
import torch
import torch.nn as nn

class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super().__init__()
        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size=kernel_size, stride=stride, padding=padding, groups=in_channels)
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.batchnorm = nn.BatchNorm2d(out_channels)
        self.activation = nn.ReLU(inplace=True)
    
    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.batchnorm(x)
        return self.activation(x)

# Inverted Residual
- Expand -> Depthwise -> Project
    |----- Skip Connection --|
- Linear final projection layer (Why? Avoid information loss in low dimensional)
- Useful in edge devices
- If expansion ratio too small -> low accuracy, too large -> hurts latency
- Skip connection if stride & channel dimensions match

In [ ]:
import torch
import torch.nn as nn

class InvertedResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super().__init__()
        mid_channels = in_channels * 6
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=1),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, mid_channels, kernel_size=kernel_size, stride=stride, padding=padding, groups=mid_channels),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=1),
            nn.BatchNorm2d(out_channels)
        )
        self.use_shortcut = (in_channels == out_channels and stride == 1)
    
    def forward(self, x):
        return self.layers(x) + (x if self.use_shortcut else 0)

# Attention Block
- Squeeze-and-Excitation Block
- B x C x H x W -- GAP (Global Average Pooling) -> B x C -> FC -> alpha
- Y_c = alpha_c \dot X_c
- Parameters: C x C/r + C/r x C
- Increase accuracy, lightweight
- Add small overhead